# WikiText Translation Dataset
Builds three datasets by translating English WikiText sequences into 8 languages.
All tokenization uses tiktoken o200k_base.
- `wikitext_translation_english` — original English (100 tokens)
- `wikitext_translation_latin` — Spanish, Portuguese, French, Italian (100 tokens)
- `wikitext_translation_diverse` — Russian, Chinese (Simplified), Tamil, Arabic (100 tokens)

# Section 1: Load & Filter WikiText

In [ ]:
import hashlib
import json
import os
import re
import time

import tiktoken
from datasets import Dataset, load_dataset
from dotenv import load_dotenv
from openai import OpenAI
from tqdm.notebook import tqdm

OPENAI_MODEL     = "gpt-5-mini-2025-08-07"
PREFIX_LENGTH    = 50
SUFFIX_LENGTH    = 50
SEQ_LEN          = PREFIX_LENGTH + SUFFIX_LENGTH   # 100
MIN_TOKEN_LENGTH = 2 * SEQ_LEN                     # 200
NUM_SEQUENCES    = 2000
SEED             = 42
CACHE_PATH       = "translation_cache.json"

LATIN_LANGUAGES   = ["Spanish", "Portuguese", "French", "Italian"]
DIVERSE_LANGUAGES = ["Russian", "Chinese (Simplified)", "Tamil", "Arabic"]
ALL_LANGUAGES     = LATIN_LANGUAGES + DIVERSE_LANGUAGES

load_dotenv()
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
enc    = tiktoken.get_encoding("o200k_base")

In [ ]:
def _prepare_source_dataset(seed: int = SEED):
    """Load, filter headers, and shuffle the WikiText dataset."""
    dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")
    pattern = r"(?: =)+[^=]+(?:= )+"
    dataset = dataset.filter(
        lambda x: not re.match(pattern, x["text"]),
        desc="Filtering headers",
        num_proc=16,
    )
    return dataset.shuffle(seed=seed)


dataset   = _prepare_source_dataset()
sequences = []

for example in tqdm(dataset, desc="Collecting source sequences"):
    if len(sequences) >= NUM_SEQUENCES:
        break
    tokens = enc.encode(example["text"])
    if len(tokens) < MIN_TOKEN_LENGTH:
        continue
    # Decode 2*SEQ_LEN tokens as the source text to translate
    source_text = enc.decode(tokens[:MIN_TOKEN_LENGTH])
    sequences.append({
        "seq_idx":     len(sequences),
        "english_ids": tokens[:SEQ_LEN],
        "source_text": source_text,
    })

print(f"Collected {len(sequences)} source sequences")

# Section 2: Translate Sequences

In [ ]:
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, encoding="utf-8") as f:
        cache = json.load(f)
else:
    cache = {}


def translate_all(text: str, languages: list[str]) -> dict[str, str]:
    """Translate text into all languages in one API call. Returns {language: translation}."""
    sha = hashlib.sha256(text.encode()).hexdigest()
    keys = {lang: f"{OPENAI_MODEL}:{lang}:{sha}" for lang in languages}

    # Return from cache if all languages already translated
    cached = {lang: cache[key] for lang, key in keys.items() if key in cache}
    missing = [lang for lang in languages if lang not in cached]

    if not missing:
        return cached

    lang_list = ", ".join(missing)
    prompt = (
        f"Translate the following text into each of these languages: {lang_list}.\n"
        f"Return ONLY a JSON object where each key is the language name and the value is the translation.\n\n"
        f"Text: {text}"
    )

    for attempt in range(5):
        try:
            resp = client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
            )
            result = json.loads(resp.choices[0].message.content)
            missing_keys = [lang for lang in missing if lang not in result]
            if missing_keys:
                raise ValueError(f"Response missing languages: {missing_keys}")
            # Cache each result individually
            for lang in missing:
                cache[keys[lang]] = result[lang]
            with open(CACHE_PATH, "w", encoding="utf-8") as f:
                json.dump(cache, f, ensure_ascii=False, indent=2)
            return {**cached, **{lang: result[lang] for lang in missing}}
        except Exception as e:
            time.sleep(2 ** attempt)
    raise RuntimeError(f"Failed to translate into {missing} after 5 retries")

In [ ]:
rows_english = []
rows_latin   = []
rows_diverse = []
kept_idx     = 0

for seq in tqdm(sequences, desc="Translating"):
    # Translate all 8 languages in a single API call; skip if any produces < SEQ_LEN tokens
    translations = translate_all(seq["source_text"], ALL_LANGUAGES)
    translated_ids: dict[str, list[int]] = {}
    skip = False
    for lang in ALL_LANGUAGES:
        ids = enc.encode(translations[lang])
        if len(ids) < SEQ_LEN:
            print(f"seq_idx={seq['seq_idx']}: {lang} only produced {len(ids)} tokens — skipping")
            skip = True
            break
        translated_ids[lang] = ids[:SEQ_LEN]

    if skip:
        continue

    # English row
    rows_english.append({
        "input_ids":      seq["english_ids"],
        "attention_mask": [1] * SEQ_LEN,
        "language":       "English",
        "group":          "english",
        "source_idx":     kept_idx,
    })

    # Latin rows
    for lang in LATIN_LANGUAGES:
        rows_latin.append({
            "input_ids":      translated_ids[lang],
            "attention_mask": [1] * SEQ_LEN,
            "language":       lang,
            "group":          "latin",
            "source_idx":     kept_idx,
        })

    # Diverse rows
    for lang in DIVERSE_LANGUAGES:
        rows_diverse.append({
            "input_ids":      translated_ids[lang],
            "attention_mask": [1] * SEQ_LEN,
            "language":       lang,
            "group":          "diverse",
            "source_idx":     kept_idx,
        })

    kept_idx += 1

print(f"Kept {kept_idx} sequences (skipped {len(sequences) - kept_idx})")

# Section 3: Assemble & Save Datasets

In [ ]:
ds_english = Dataset.from_list(rows_english).shuffle(seed=SEED)
ds_latin   = Dataset.from_list(rows_latin).shuffle(seed=SEED)
ds_diverse = Dataset.from_list(rows_diverse).shuffle(seed=SEED)

ds_english.save_to_disk("wikitext_translation_english")
ds_latin.save_to_disk("wikitext_translation_latin")
ds_diverse.save_to_disk("wikitext_translation_diverse")

print(f"English: {len(ds_english)} rows -> 'wikitext_translation_english'")
print(f"Latin:   {len(ds_latin)} rows -> 'wikitext_translation_latin'")
print(f"Diverse: {len(ds_diverse)} rows -> 'wikitext_translation_diverse'")

# Section 4: Validation

In [ ]:
from collections import defaultdict
from datasets import load_from_disk

ds_eng = load_from_disk("wikitext_translation_english")
ds_lat = load_from_disk("wikitext_translation_latin")
ds_div = load_from_disk("wikitext_translation_diverse")


def group_by_source(ds):
    groups = defaultdict(list)
    for row in ds:
        groups[row["source_idx"]].append(row)
    return groups


eng_groups = group_by_source(ds_eng)
lat_groups = group_by_source(ds_lat)
div_groups = group_by_source(ds_div)

all_source_idxs = sorted(eng_groups.keys() | lat_groups.keys() | div_groups.keys())

# 1. All 9 languages present per source_idx
for idx in all_source_idxs:
    eng_langs = {r["language"] for r in eng_groups[idx]}
    lat_langs = {r["language"] for r in lat_groups[idx]}
    div_langs = {r["language"] for r in div_groups[idx]}
    assert eng_langs == {"English"}, f"source_idx {idx}: English missing, got {eng_langs}"
    assert lat_langs == set(LATIN_LANGUAGES), f"source_idx {idx}: Latin langs {lat_langs} != {set(LATIN_LANGUAGES)}"
    assert div_langs == set(DIVERSE_LANGUAGES), f"source_idx {idx}: Diverse langs {div_langs} != {set(DIVERSE_LANGUAGES)}"
print(f"[PASS] All 9 languages present for each of {len(all_source_idxs)} source sequences.")

# 2. All rows have length SEQ_LEN
for ds_name, ds in [("english", ds_eng), ("latin", ds_lat), ("diverse", ds_div)]:
    bad = [i for i, row in enumerate(ds) if len(row["input_ids"]) != SEQ_LEN]
    assert not bad, f"{ds_name}: rows with wrong length: {bad[:5]}"
print(f"[PASS] All rows have input_ids of length {SEQ_LEN}.")

print(f"\nAll validations passed. ({len(ds_eng)} english, {len(ds_lat)} latin, {len(ds_div)} diverse rows)")